# Notebook 02-UCI - Models, calibration, and fairness audit (second dataset)

The same modelling and fairness protocol used on OULAD, applied to UCI 697. Three model families are
trained at each snapshot, calibrated on a held-out split, and audited for group fairness, with
bootstrap confidence intervals.

**Snapshots:** T0 (enrollment-only) and T1 (enrollment + semester 1) are the main results; T2
(+ semester 2) is computed but flagged as an appendix late-snapshot upper bound, since second-semester
fields are near-proxies for having already left.

**Labels:** primary (Dropout = 1, Graduate = 0, Enrolled removed) and fold (Dropout = 1, Graduate and
Enrolled = 0). Running the audit on both tests label-taxonomy robustness.

**Audit axes (sensitive-attribute transfer).**
* Primary, well-powered: scholarship (socioeconomic-support proxy, direction-agnostic), gender, age
  (young vs older tertile extremes).
* Underpowered, reported with its subgroup size: disability (educational special needs, n is small).
* Robustness only, outcome-proximal: debtor, financial_vulnerability.

**Inputs:** `uci_model_ready_T0/T1/T2`, `uci_features.json`. **Outputs:** `uci_metrics_summary.csv`,
`uci_fairness_summary.csv`, `uci_models_{snapshot}.joblib`, `uci_predictions_{snapshot}` (primary label).

## 0. Setup

In [1]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/StudentEWS_Research/student-ews-research')
except Exception:
    ROOT = Path('.')
PROC   = ROOT / 'results' / 'processed'
MODELS = ROOT / 'results' / 'models'; MODELS.mkdir(parents=True, exist_ok=True)

SEED = 42; B_BOOT = 1000; THRESH = 0.5
SNAPSHOTS = ['T0', 'T1', 'T2']         # T2 flagged appendix in the reporting
MODEL_ORDER = ['logreg', 'rf', 'hgb']
# axis -> (column, group_A_value, group_B_value, tier)
AXES = {
    'scholarship':    ('scholarship', 'Y', 'N', 'primary'),
    'gender':         ('gender', 'M', 'F', 'primary'),
    'age':            ('age_group', 'young', 'older', 'primary'),
    'disability':     ('disability', 'Y', 'N', 'underpowered'),
    'debtor':         ('debtor', 'Y', 'N', 'robustness'),
    'fin_vulnerable': ('fin_vulnerable', 'Y', 'N', 'robustness'),
}

Mounted at /content/drive


In [2]:
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score, recall_score, f1_score, brier_score_loss

## 1. Helpers (shared protocol with the OULAD notebooks)

In [3]:
def load(stem):
    for e in ('.parquet', '.csv'):
        if (Path(str(stem) + e)).exists():
            return pd.read_parquet(str(stem) + e) if e == '.parquet' else pd.read_csv(str(stem) + e)
    return None

def ece(y, p, n_bins=10):
    y, p = np.asarray(y, float), np.asarray(p, float)
    edges = np.linspace(0, 1, n_bins + 1)
    b = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    n = len(p); e = 0.0
    for k in range(n_bins):
        msk = b == k
        if msk.sum(): e += (msk.sum()/n) * abs(y[msk].mean() - p[msk].mean())
    return float(e)

def fit_calibrator(p, y):
    p, y = np.asarray(p, float), np.asarray(y, int)
    if len(p) >= 30 and len(np.unique(y)) > 1:
        iso = IsotonicRegression(out_of_bounds='clip'); iso.fit(p, y); return ('isotonic', iso)
    lr = LogisticRegression(); lr.fit(p.reshape(-1, 1), y); return ('platt', lr)

def apply_calibrator(cal, p):
    kind, model = cal; p = np.asarray(p, float)
    return model.predict(p) if kind == 'isotonic' else model.predict_proba(p.reshape(-1, 1))[:, 1]

def build_models():
    return {
        'logreg': Pipeline([('scaler', StandardScaler()),
                            ('clf', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=SEED))]),
        'rf': RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=SEED),
        'hgb': HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, l2_regularization=1.0, random_state=SEED),
    }

def _rates(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    flag = pred.mean() if len(pred) else np.nan
    tpr = pred[y == 1].mean() if (y == 1).any() else np.nan
    fpr = pred[y == 0].mean() if (y == 0).any() else np.nan
    return flag, tpr, fpr

def fairness_gaps(yA, pA, yB, pB, B, seed):
    yA, pA, yB, pB = map(np.asarray, (yA, pA, yB, pB))
    fA, fB = _rates(yA, pA), _rates(yB, pB)
    raw = np.array(fA) - np.array(fB)
    r = np.random.default_rng(seed); nA, nB = len(yA), len(yB)
    boot = np.empty((B, 3))
    for i in range(B):
        ia, ib = r.integers(0, nA, nA), r.integers(0, nB, nB)
        boot[i] = np.array(_rates(yA[ia], pA[ia])) - np.array(_rates(yB[ib], pB[ib]))
    ci = np.nanpercentile(boot, [2.5, 97.5], axis=0)
    return fA, fB, raw, ci

FEATS = json.load(open(ROOT / 'results' / 'uci_features.json'))

## 2. Train, calibrate, score, and audit

In [4]:
metrics_rows, fair_rows = [], []
for snap in SNAPSHOTS:
    features = FEATS[snap]
    ready = load(PROC / f'uci_model_ready_{snap}')
    for label in ['primary', 'fold']:
        if label == 'primary':
            data = ready[ready['enrolled_flag'] == 0].copy(); ycol = 'at_risk'
        else:
            data = ready.copy(); ycol = 'at_risk_fold'
        y = data[ycol].astype(int)
        tr = (data['split'] == 'train'); ca = (data['split'] == 'calib'); te = (data['split'] == 'test')
        X = data[features].apply(pd.to_numeric, errors='coerce')
        med = X[tr].median()
        X = X.fillna(med)

        bundle = {'features': features, 'models': {}}
        preds = data.loc[te, ['student_id','scholarship','gender','age_group','disability',
                              'debtor','fin_vulnerable', ycol]].copy().reset_index(drop=True)
        for name, est in build_models().items():
            if name == 'hgb':
                est.fit(X[tr], y[tr], sample_weight=compute_sample_weight('balanced', y[tr]))
            else:
                est.fit(X[tr], y[tr])
            cal = fit_calibrator(est.predict_proba(X[ca])[:, 1], y[ca])
            p_un = est.predict_proba(X[te])[:, 1]; p_cal = apply_calibrator(cal, p_un)
            yte = y[te].to_numpy(); pred = (p_cal >= THRESH).astype(int)
            metrics_rows.append({
                'dataset': 'UCI', 'snapshot': snap, 'label': label, 'model': name,
                'appendix': snap == 'T2', 'positive_rate': float(y[tr].mean()), 'n_test': int(te.sum()),
                'auroc': roc_auc_score(yte, p_un), 'recall': recall_score(yte, pred, zero_division=0),
                'f1': f1_score(yte, pred, zero_division=0), 'brier_cal': brier_score_loss(yte, p_cal),
                'ece_uncal': ece(yte, p_un), 'ece_cal': ece(yte, p_cal),
            })
            if label == 'primary':
                bundle['models'][name] = {'estimator': est, 'calibrator': cal}
                preds[f'p_{name}'] = p_cal; preds[f'pred_{name}'] = pred

            # fairness audit
            for axis, (col, vA, vB, tier) in AXES.items():
                mA = (data.loc[te, col] == vA).to_numpy(); mB = (data.loc[te, col] == vB).to_numpy()
                if mA.sum() == 0 or mB.sum() == 0: continue
                fA, fB, raw, ci = fairness_gaps(yte[mA], pred[mA], yte[mB], pred[mB], B_BOOT, SEED)
                fair_rows.append({
                    'dataset': 'UCI', 'snapshot': snap, 'label': label, 'model': name, 'axis': axis,
                    'tier': tier, 'appendix': snap == 'T2', 'n_A': int(mA.sum()), 'n_B': int(mB.sum()),
                    'group_A': vA, 'group_B': vB,
                    'dp_gap': raw[0], 'eo_gap': raw[1], 'eo_lo': ci[0,1], 'eo_hi': ci[1,1],
                    'fpr_gap': raw[2], 'fpr_lo': ci[0,2], 'fpr_hi': ci[1,2],
                })
        if label == 'primary':
            joblib.dump(bundle, MODELS / f'uci_models_{snap}.joblib')
            try: preds.to_parquet(PROC / f'uci_predictions_{snap}.parquet', index=False)
            except Exception: preds.to_csv(PROC / f'uci_predictions_{snap}.csv', index=False)
    print(f'{snap}: done')

uci_metrics = pd.DataFrame(metrics_rows); uci_metrics.to_csv(ROOT / 'results' / 'uci_metrics_summary.csv', index=False)
uci_fair = pd.DataFrame(fair_rows); uci_fair.to_csv(ROOT / 'results' / 'uci_fairness_summary.csv', index=False)
print('saved uci_metrics_summary.csv and uci_fairness_summary.csv')

T0: done
T1: done
T2: done
saved uci_metrics_summary.csv and uci_fairness_summary.csv


## 3. Performance and calibration (main snapshots, primary label)

In [5]:
main = uci_metrics[(uci_metrics.label == 'primary') & (~uci_metrics.appendix)]
print(main[['snapshot','model','positive_rate','auroc','recall','ece_uncal','ece_cal']]
      .round(4).to_string(index=False))
print('\nAppendix (T2, primary) - late-snapshot upper bound, interpret with leakage caution:')
print(uci_metrics[(uci_metrics.label == 'primary') & (uci_metrics.appendix)]
      [['snapshot','model','auroc','recall','ece_cal']].round(4).to_string(index=False))

snapshot  model  positive_rate  auroc  recall  ece_uncal  ece_cal
      T0 logreg         0.3916 0.7622  0.4824     0.0862   0.0289
      T0     rf         0.3916 0.7489  0.5317     0.0415   0.0271
      T0    hgb         0.3916 0.7406  0.4296     0.0870   0.0429
      T1 logreg         0.3916 0.9199  0.7183     0.0524   0.0391
      T1     rf         0.3916 0.9074  0.7394     0.0720   0.0354
      T1    hgb         0.3916 0.9066  0.7676     0.0566   0.0237

Appendix (T2, primary) - late-snapshot upper bound, interpret with leakage caution:
snapshot  model  auroc  recall  ece_cal
      T2 logreg 0.9428  0.8028   0.0305
      T2     rf 0.9402  0.7887   0.0219
      T2    hgb 0.9362  0.7606   0.0252


## 4. Fairness audit (primary label, main snapshots)

In [6]:
fp = uci_fair[(uci_fair.label == 'primary') & (~uci_fair.appendix)]
print('FPR gap (group A minus B) by axis, gradient boosting:')
print(fp[fp.model == 'hgb'][['snapshot','axis','tier','n_A','n_B','fpr_gap','fpr_lo','fpr_hi']]
      .round(4).to_string(index=False))
print('\nReminders for interpretation:')
print('- scholarship is direction-agnostic (holders are lower-risk here, opposite to OULAD deprivation).')
print('- disability n_A is small; treat any null on that axis as underpowered, not as evidence of equity.')
print('- debtor and fin_vulnerable are outcome-proximal; robustness only, not protected-background axes.')

# does the headline FPR finding survive the fold label? (scholarship, hgb)
fold = uci_fair[(uci_fair.label == 'fold') & (~uci_fair.appendix) &
                (uci_fair.model == 'hgb') & (uci_fair.axis == 'scholarship')]
print('\nScholarship FPR gap under the fold label (label-taxonomy robustness, hgb):')
print(fold[['snapshot','fpr_gap','fpr_lo','fpr_hi']].round(4).to_string(index=False))

FPR gap (group A minus B) by axis, gradient boosting:
snapshot           axis         tier  n_A  n_B  fpr_gap  fpr_lo  fpr_hi
      T0    scholarship      primary  199  527  -0.0710 -0.1313 -0.0081
      T0         gender      primary  251  475  -0.0289 -0.0995  0.0433
      T0            age      primary  313  269  -0.3343 -0.4304 -0.2397
      T0     disability underpowered    7  719   0.1199 -0.1554  0.6746
      T0         debtor   robustness   99  627   0.0180 -0.1069  0.1690
      T0 fin_vulnerable   robustness  152  574  -0.0024 -0.1217  0.1187
      T1    scholarship      primary  199  527  -0.0618 -0.1132 -0.0145
      T1         gender      primary  251  475  -0.0018 -0.0669  0.0646
      T1            age      primary  313  269  -0.0879 -0.1713 -0.0153
      T1     disability underpowered    7  719   0.1610 -0.1102  0.7377
      T1         debtor   robustness   99  627   0.0220 -0.0858  0.1538
      T1 fin_vulnerable   robustness  152  574   0.0414 -0.0724  0.1780

Reminders

## What was saved, and what comes next

* `uci_metrics_summary.csv` - AUROC, recall, Brier, ECE per snapshot x label x model, with the T2 rows
  flagged `appendix`.
* `uci_fairness_summary.csv` - demographic-parity, equal-opportunity, and false-positive-rate gaps with
  bootstrap CIs, per snapshot x label x model x axis, with each axis tagged primary / underpowered /
  robustness and carrying its subgroup sizes.
* `uci_models_{snapshot}.joblib` and `uci_predictions_{snapshot}` (primary label) for the next notebook.

This delivers cross-granularity (T0 vs T1), label-taxonomy robustness (primary vs fold), and the
fairness arm of sensitive-attribute transfer. The next UCI notebook adds the methodological-finding
transfer - whether the naive subgroup faithfulness gap appears and then vanishes under base-rate
adjustment on this dataset too - plus recourse at T1 and the cross-dataset comparison with OULAD.

Paste me the section 3 and 4 printouts and I will read the second-dataset results with you.